In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import numpy as np
import re

In [2]:
options = Options()

options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
# load product href csvs
red_df = pd.read_csv("data/Red_wine_hrefs.csv")
white_df = pd.read_csv("data/White_wine_hrefs.csv")
sparkling_df = pd.read_csv("data/Champagne-Sparkling_wine_hrefs.csv")

# Combine the DataFrames into a single DataFrame
wine_df = pd.concat([red_df, white_df, sparkling_df], ignore_index=True)

# Going to Scrape:
### product description x
### type (of grapes), separated by space x
### vintage x
### state x
### region x
### wine sweetness
### wine body
### winery name x
### product name x
### non-member price x
### member price x

In [ ]:
%%time

hrefs_subset= []
product_names = []
colours = []
product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []

total_wines = len(wine_df)
starting_i = 9
try:
    for i in range(starting_i, len(wine_df), 1):
        
        # get href
        href = wine_df.iloc[i]["href"]
        #hrefs_subset.append(href)
        driver.get(href)
    
        #colours.append(wine_df.iloc[i]["type"])
        
        is_loaded = False
        try:
            # expand specifications list
            wait = WebDriverWait(driver, 10)
            expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
            expansion_panel.click()
            is_loaded = True
        except TimeoutException:
            print("!!! no product page for i =", i, "--href", href)
    
        if is_loaded:    
            # Scrape product name
            product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
        
        
            # scrape product description
            product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
            
            
            # scrape specifications list
            specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
            specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
            specifications = {}
            for li in specification_li_elements:
                key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
                value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
                specifications[key] = value
            
            if "Type" in specifications:
                types.append(specifications["Type"])
            else:
                types.append(np.nan)
            
            if "Current Vintage" in specifications:
                vintages.append(specifications["Current Vintage"])
            else:
                vintages.append(np.nan)
            
            if "State" in specifications:
                states.append(specifications["State"])
            else:
                states.append(np.nan)
            
            if "Region" in specifications:
                regions.append(specifications["Region"])
            else:
                regions.append(np.nan)
            
            if "Wine Maker" in specifications:
                wineries.append(specifications["Wine Maker"])
            else:
                wineries.append(np.nan)
            
            if "Size" in specifications:
                sizes.append(specifications["Size"])
            else:
                sizes.append(np.nan)
        
            if "Wine Sweetness" in specifications:
                sweetnesses.append(specifications["Wine Sweetness"])
            else:
                sweetnesses.append(np.nan)
        
            if "Wine Body" in specifications:
                bodies.append(specifications["Wine Body"])
            else:
                bodies.append(np.nan)
        
        
            # Scrape prices
            pattern = r'\d+\.*\d*' # regex pattern
            price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
            try:
                non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
                non_member_price = re.search(pattern, non_member_price).group()
                member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                member_price = re.search(pattern, member_price).group()
            except NoSuchElementException:
                try:
                    non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                    non_member_price = re.search(pattern, non_member_price).group()
                    member_price = np.nan
                    print("no member price")
                except NoSuchElementException:
                    non_member_price = np.nan
                    member_price = np.nan
                    print("no price found for: i =", i, href)
            
            non_member_prices.append(non_member_price)
            member_prices.append(member_price)
        
        else:
            product_names.append(np.nan)
            product_descriptions.append(np.nan)
            types.append(np.nan)
            vintages.append(np.nan)
            states.append(np.nan)
            regions.append(np.nan)
            wineries.append(np.nan)
            sizes.append(np.nan)
            non_member_prices.append(np.nan)
            member_prices.append(np.nan)
            bodies.append(np.nan)
            sweetnesses.append(np.nan)  
            
        hrefs_subset.append(href)
        colours.append(wine_df.iloc[i]["type"])
        print("scraped i =", i, "of", total_wines-1)

except Exception as e:
    print("Stopped scraping at i =", i)
    print(e)
    

no member price
scraped i = 9 of 6546
no member price
scraped i = 10 of 6546
no member price
scraped i = 11 of 6546
scraped i = 12 of 6546
scraped i = 13 of 6546
no member price
scraped i = 14 of 6546
no member price
scraped i = 15 of 6546
no member price
scraped i = 16 of 6546
no member price
scraped i = 17 of 6546
no member price
scraped i = 18 of 6546
scraped i = 19 of 6546
no member price
scraped i = 20 of 6546
no member price
scraped i = 21 of 6546
no member price
scraped i = 22 of 6546


In [ ]:
populated_wine_df = {
    "href" : hrefs_subset,
    "product_name" : product_names,
    "product_description" : product_descriptions,
    "colour" : colours,
    "type" : types,
    "vintage" : vintages,
    "state" : states,
    "region" : regions,
    "winery" : wineries,
    "size" : sizes,
    "non_member_price" : non_member_prices,
    "member_price" : member_prices,
    "body" : bodies,
    "sweetness" : sweetnesses
}

populated_wine_df = pd.DataFrame(populated_wine_df)

In [ ]:
populated_wine_df

In [ ]:
populated_wine_df.iloc[0]

In [ ]:
# save df to csv
file_path = f"data/all_wine_data_subsets/wine_dataset_i_{starting_i}-{i-1}.csv"
#populated_wine_df.to_csv(file_path, index=False)
populated_wine_df.to_csv(file_path, index=False)

## Debugging

In [ ]:
#href

In [ ]:
"""options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
product_names = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []


# get href
driver.get(href)

is_loaded = False
try:
    # expand specifications list
    wait = WebDriverWait(driver, 10)
    expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
    expansion_panel.click()
    is_loaded = True
except TimeoutException:
    print("!!! no product page for:", href)  

if is_loaded:
    # scrape product description
    product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
    
    
    # scrape specifications list
    specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
    specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
    specifications = {}
    for li in specification_li_elements:
        key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
        value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
        specifications[key] = value
    
    if "Type" in specifications:
        types.append(specifications["Type"])
    else:
        types.append(np.nan)
    
    if "Current Vintage" in specifications:
        vintages.append(specifications["Current Vintage"])
    else:
        vintages.append(np.nan)
    
    if "State" in specifications:
        states.append(specifications["State"])
    else:
        states.append(np.nan)
    
    if "Region" in specifications:
        regions.append(specifications["Region"])
    else:
        regions.append(np.nan)
    
    if "Wine Maker" in specifications:
        wineries.append(specifications["Wine Maker"])
    else:
        wineries.append(np.nan)
    
    if "Size" in specifications:
        sizes.append(specifications["Size"])
    else:
        sizes.append(np.nan)
    
    
    # Scrape product name
    product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
    
    
    # Scrape prices
    pattern = r'\d+\.*\d*' # regex pattern
    price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
    try:
        non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
        non_member_price = re.search(pattern, non_member_price).group()
        member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
        member_price = re.search(pattern, member_price).group()
    except NoSuchElementException:
        try:
            non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
            non_member_price = re.search(pattern, non_member_price).group()
            member_price = np.nan
            print("no member price")
        except NoSuchElementException:
            non_member_price = np.nan
            member_price = np.nan
            print("no price found for:", href)
    
    non_member_prices.append(non_member_price)
    member_prices.append(member_price)   
    
    # Scrape body and sweetness values
    profile_bars = driver.find_elements(By.CLASS_NAME, "body-profile__bar")
    body_bar = profile_bars[0]
    sweetness_bar = profile_bars[1]
    body_indicators = body_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
    sweetness_indicators = sweetness_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
    
    def get_indicator_value(indicators):
        for index, indicator in enumerate(indicators):
            opacity = indicator.value_of_css_property('opacity')
            if opacity == '1':
                index_with_opacity_1 = index
                return index_with_opacity_1
            else:
                return np.nan
    
    body = get_indicator_value(body_indicators)
    bodies.append(body)
    sweetness = get_indicator_value(sweetness_indicators)
    sweetnesses.append(sweetness)

else:
    product_names.append(np.nan)
    product_descriptions.append(np.nan)
    types.append(np.nan)
    vintages.append(np.nan)
    states.append(np.nan)
    regions.append(np.nan)
    wineries.append(np.nan)
    sizes.append(np.nan)
    non_member_prices.append(np.nan)
    member_prices.append(np.nan)
    bodies.append(np.nan)
    sweetnesses.append(np.nan)"""